### 1. Data Loading and EDA

In [1]:
import pandas as pd

# Load dataset from Excel file
df = pd.read_excel('datathon-dataset-2026.xlsx', sheet_name='data')

In [2]:
# Find number of rows and columns in the dataset
nrow, ncol = df.shape
print(f"The dataset has {nrow} rows and {ncol} columns.")

The dataset has 69557 rows and 67 columns.


In [3]:
# Display the first few rows of the dataset
display(df.head())

,student_id,entry_year,credit_hrs,app_dt,app_mo,app_yr,app_fee,acceptance_fee,hs_gpa,act_cmpst,...,hs_only_math,hs_only_sci,rotc_interest,miles_from_campus,veteran,age,stem_major,urm,sex,college
0,145382.0,2022,0.0,13.0,nov,2021,Deferred,NaN,2.15,11.0,...,8.0,8.0,0,855.0,0,20.0,0,1,M,Public Health and Translational Science
1,89694.0,2022,0.0,27.0,dec,2021,Deferred,NaN,2.14,11.0,...,NaN,4.0,0,307.0,0,18.0,0,1,M,Public Health and Translational Science
2,112761.0,2022,0.0,27.0,nov,2021,Deferred,NaN,2.67,11.0,...,NaN,NaN,0,534.0,0,19.0,1,1,F,Public Health and Translational Science
3,61545.0,2022,0.0,11.0,oct,2021,Deferred,NaN,2.84,11.0,...,8.0,7.0,0,29.0,0,18.0,0,0,M,Liberal Arts and Sciences
4,113888.0,2022,0.0,9.0,aug,2021,Paid,NaN,2.22,12.0,...,NaN,NaN,0,226.0,0,18.0,0,1,M,"Business, Finance and Industry"


In [19]:
# Check number and ratio of male and female applicants to each college in the dataset
mf_applicants = df.groupby(['college', 'sex']).size().unstack(fill_value=0).assign(total=lambda x: (x['M'] + x['F'])).reset_index()
mf_applicants['male_ratio'] = mf_applicants['M'] / mf_applicants['total']
mf_applicants['female_ratio'] = mf_applicants['F'] / mf_applicants['total']
mf_applicants.loc[len(mf_applicants)] = ['Total', mf_applicants['M'].sum(), mf_applicants['F'].sum(), mf_applicants['total'].sum(), mf_applicants['M'].sum() / mf_applicants['total'].sum(), mf_applicants['F'].sum() / mf_applicants['total'].sum()]
mf_applicants

sex,college,F,M,total,male_ratio,female_ratio
0,Applied Science and Engineering,13971,3624,17595,0.205968,0.794032
1,"Business, Finance and Industry",5439,3268,8707,0.375330,0.624670
2,"Land, Environment and Food Sciences",3045,5212,8257,0.631222,0.368778
3,Liberal Arts and Sciences,10311,13414,23725,0.565395,0.434605
4,Performing and Fine Arts,1739,2247,3986,0.563723,0.436277
5,Public Health and Translational Science,1755,5532,7287,0.759160,0.240840
6,Total,33297,36260,69557,0.478701,0.521299


In [25]:
# Check number of applicants by residency status (in-state, contiguous, non-contiguous) to each college in the dataset
residency_applicants = df.groupby(['college', 'residency']).size().unstack(fill_value=0).assign(total=lambda x: (x['Resident'] + x['Contiguous'] + x['Non-Contiguous'])).reset_index()
residency_applicants['resident_ratio'] = residency_applicants['Resident'] / residency_applicants['total']
residency_applicants['contiguous_ratio'] = residency_applicants['Contiguous'] / residency_applicants['total']
residency_applicants['non_contiguous_ratio'] = residency_applicants['Non-Contiguous'] / residency_applicants['total']
residency_applicants.loc[len(residency_applicants)] = ['Total', residency_applicants['Resident'].sum(), residency_applicants['Contiguous'].sum(), residency_applicants['Non-Contiguous'].sum(), residency_applicants['total'].sum(), residency_applicants['Resident'].sum() / residency_applicants['total'].sum(), residency_applicants['Contiguous'].sum() / residency_applicants['total'].sum(), residency_applicants['Non-Contiguous'].sum() / residency_applicants['total'].sum()]
residency_applicants

residency,college,Contiguous,Non-Contiguous,Resident,total,resident_ratio,contiguous_ratio,non_contiguous_ratio
0,Applied Science and Engineering,10176,2658,4761,17595,0.270588,0.578346,0.151066
1,"Business, Finance and Industry",4906,989,2812,8707,0.322959,0.563455,0.113587
2,"Land, Environment and Food Sciences",3344,1653,3260,8257,0.394817,0.404990,0.200194
3,Liberal Arts and Sciences,11318,3493,8914,23725,0.375722,0.477050,0.147229
4,Performing and Fine Arts,1848,644,1494,3986,0.374812,0.463623,0.161565
5,Public Health and Translational Science,3298,1042,2947,7287,0.404419,0.452587,0.142994
6,Total,24188,34890,10479,69557,0.347744,0.501603,0.150653


In [28]:
# Check admit and deposit rates for each college in the dataset
admit_deposit_rates = df.groupby('college').agg({'admitted': 'sum', 'deposited': 'sum'}).reset_index()
admit_deposit_rates['admit_rate'] = admit_deposit_rates['admitted'] / df.groupby('college').size().values
admit_deposit_rates['deposit_rate'] = admit_deposit_rates['deposited'] / df.groupby('college').size().values
admit_deposit_rates.loc[len(admit_deposit_rates)] = ['Total', admit_deposit_rates['admitted'].sum(), admit_deposit_rates['deposited'].sum(), admit_deposit_rates['admitted'].sum() / df.shape[0], admit_deposit_rates['deposited'].sum() / df.shape[0]]
admit_deposit_rates

,college,admitted,deposited,admit_rate,deposit_rate
0,Applied Science and Engineering,16823,6902,0.956124,0.392271
1,"Business, Finance and Industry",7487,2602,0.859883,0.298840
2,"Land, Environment and Food Sciences",7456,3268,0.902991,0.395785
3,Liberal Arts and Sciences,20560,6376,0.866596,0.268746
4,Performing and Fine Arts,3565,1559,0.894380,0.391119
5,Public Health and Translational Science,6181,2500,0.848223,0.343077
6,Total,62072,23207,0.892390,0.333640


In [31]:
# Find average age, high school GPA, ACT composite score, and SAT composite score for each college in the dataset
average_stats = df.groupby('college').agg({'age': 'mean', 'hs_gpa': 'mean', 'act_cmpst': 'mean', 'sat_comb': 'mean'}).reset_index()
average_stats.loc[len(average_stats)] = ['Total', df['age'].mean(), df['hs_gpa'].mean(), df['act_cmpst'].mean(), df['sat_comb'].mean()]
average_stats

,college,age,hs_gpa,act_cmpst,sat_comb
0,Applied Science and Engineering,18.427816,3.660323,27.204506,1246.758509
1,"Business, Finance and Industry",18.476108,3.338055,23.504051,1096.443149
2,"Land, Environment and Food Sciences",18.441323,3.549007,24.001276,1116.933216
3,Liberal Arts and Sciences,18.431697,3.425539,24.252349,1135.250000
4,Performing and Fine Arts,18.455344,3.486120,24.251253,1159.872340
5,Public Health and Translational Science,18.444765,3.401952,22.562964,1072.687500
6,Total,18.440141,3.489686,24.725064,1160.790823


In [37]:
# Check academic performance of applicants by residency status (in-state, contiguous, non-contiguous)
academic_performance = df.groupby(['residency']).agg({'hs_gpa': 'mean', 'act_cmpst': 'mean', 'sat_comb': 'mean'}).reset_index()

# Add columns for number of applicants that submitted ACT and SAT scores
academic_performance['num_applicants'] = df.groupby('residency').size().values
academic_performance['act_submitted'] = df[df['act_cmpst'].notnull()].groupby('residency').size().values
academic_performance['sat_submitted'] = df[df['sat_comb'].notnull()].groupby('residency').size().values
academic_performance['act_submission_rate'] = academic_performance['act_submitted'] / df.groupby('residency').size().values
academic_performance['sat_submission_rate'] = academic_performance['sat_submitted'] / df.groupby('residency').size().values
academic_performance.loc[len(academic_performance)] = ['Total', df['hs_gpa'].mean(), df['act_cmpst'].mean(), df['sat_comb'].mean(), sum(academic_performance['num_applicants']), df[df['act_cmpst'].notnull()].shape[0], df[df['sat_comb'].notnull()].shape[0], df[df['act_cmpst'].notnull()].shape[0] / df.shape[0], df[df['sat_comb'].notnull()].shape[0] / df.shape[0]]
academic_performance

,residency,hs_gpa,act_cmpst,sat_comb,num_applicants,act_submitted,sat_submitted,act_submission_rate,sat_submission_rate
0,Contiguous,3.477369,25.384492,1189.248536,34890,25574,3074,0.732989,0.088105
1,Non-Contiguous,3.489747,24.933745,1123.862959,10479,4211,4028,0.401851,0.384388
2,Resident,3.507470,23.895022,1248.314286,24188,21376,700,0.883744,0.028940
3,Total,3.489686,24.725064,1160.790823,69557,51161,7802,0.735526,0.112167


In [5]:
# Check for duplicate rows in the dataset
int(df.duplicated().sum())

0

In [6]:
# Find columns with most null values in the dataset; exclude target variable 'enrolled' from the null value check
df.isnull().sum().drop('enrolled', errors='ignore').sort_values(ascending=False).head(20)

sat_math            61755
sat_comb            61755
sat_vrbl            61755
deposit_dt          44085
deposit_mo          43932
acceptance_fee      43587
deposit_yr          43326
visit_mo            37153
visit_yr            37153
visit_dt            37153
act_engl            18493
act_math            18492
act_cmpst           18396
hs_only_sci         18000
hs_flang            17075
hs_sci              16955
hs_only_math        11689
hs_ss               11307
hs_math             10606
first_contact_dt     2509
dtype: int64

It seems as if there is almost no information information about SAT scores, and not a lot of information about dates for admission offer acceptances and campus visits. Comparatively to SAT scores, even though there is missing information about ACT scores, the amount present is significantly more than for SAT scores, which makes sense because students more commonly take the ACT as a standardized test over the SAT.